## Google Colab and data paths

Set `KDD_PROJECT_ROOT` if your project is not stored at `/content/drive/MyDrive/kdd_dropout_project`. The configured folder must contain `dataset/`, `shared/metadata/`, and `artifacts/`.


In [ ]:
from pathlib import Path
import os

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    pass

def discover_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset").exists() and (candidate / "shared" / "metadata").exists():
            return candidate
    return start

default_root = (
    Path("/content/drive/MyDrive/kdd_dropout_project")
    if IN_COLAB else discover_project_root(Path.cwd())
)
PROJECT_ROOT = Path(os.environ.get("KDD_PROJECT_ROOT", default_root))
DATA_ROOT = PROJECT_ROOT / "dataset"
METADATA_ROOT = PROJECT_ROOT / "shared" / "metadata"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
MODEL_ARTIFACT_ROOT = ARTIFACT_ROOT / "baseline_bilstm"
MODEL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Model artifacts: {MODEL_ARTIFACT_ROOT}")


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ==========================================
# 1. LOAD PROCESSED DATA
# ==========================================
print("Loading processed data...")
data = torch.load(MODEL_ARTIFACT_ROOT / "processed_data.pt", weights_only=False)  # Trust only the artifact created by notebook 02

src_train, evt_train, cat_train, gap_train, y_train = data["train"]
src_test, evt_test, cat_test, gap_test, y_test, test_ids = data["test"]
source_vocab, event_vocab, category_vocab = data["vocabs"]
truth_test = data["truth_test"]

class SeqDataset(Dataset):
    def __init__(self, src, evt, cat, gaps, y):
        self.src, self.evt, self.cat = torch.LongTensor(src), torch.LongTensor(evt), torch.LongTensor(cat)
        self.gaps, self.y = torch.FloatTensor(gaps), torch.FloatTensor(y)

    def __len__(self): return len(self.src)
    def __getitem__(self, idx): return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx], self.y[idx]

train_loader = DataLoader(SeqDataset(src_train, evt_train, cat_train, gap_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(SeqDataset(src_test, evt_test, cat_test, gap_test, y_test), batch_size=128)

# ==========================================
# 2. DEFINE MODEL
# ==========================================
class BiLSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.source_emb = nn.Embedding(len(source_vocab) + 1, 4, padding_idx=0)
        self.event_emb = nn.Embedding(len(event_vocab) + 1, 8, padding_idx=0)
        self.category_emb = nn.Embedding(len(category_vocab) + 1, 8, padding_idx=0)
        
        self.lstm = nn.LSTM(input_size=21, hidden_size=64, batch_first=True, bidirectional=True)
        self.attention = nn.Linear(128, 1)
        self.fc = nn.Linear(128, 1)

    def forward(self, src, evt, cat, gaps, lengths):
        x = torch.cat([self.source_emb(src), self.event_emb(evt), self.category_emb(cat), gaps.unsqueeze(-1)], dim=2)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = torch.nn.utils.rnn.pad_packed_sequence(self.lstm(packed)[0], batch_first=True)
        
        attn_scores = self.attention(out).squeeze(-1)
        attn_scores[~(src[:, :out.size(1)] != 0)] = -1e9 # Mask padding
        context = torch.sum(out * torch.softmax(attn_scores, dim=1).unsqueeze(-1), dim=1)
        return torch.sigmoid(self.fc(context)).squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMModel().to(device)

# ==========================================
# 3. TRAINING & EVALUATION
# ==========================================
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

print(f"Training on Device: {device}...")
for epoch in range(3):
    model.train()
    total_loss = 0
    for src, evt, cat, gaps, labels in train_loader:
        src, evt, cat, gaps, labels = [t.to(device) for t in (src, evt, cat, gaps, labels)]
        
        optimizer.zero_grad()
        outputs = model(src, evt, cat, gaps, (src != 0).sum(dim=1))
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

print("\nEvaluating Test Data...")
model.eval()
window_preds = []
with torch.no_grad():
    for src, evt, cat, gaps, _ in test_loader:
        src, evt, cat, gaps = [t.to(device) for t in (src, evt, cat, gaps)]
        outputs = model(src, evt, cat, gaps, (src != 0).sum(dim=1))
        window_preds.extend(outputs.cpu().numpy())

# Calculate metrics
pred_df = pd.DataFrame({"student_id": test_ids, "pred": window_preds})
eval_df = pred_df.groupby("student_id")["pred"].mean().reset_index().merge(truth_test, on="student_id")
eval_df["predicted_dropout"] = (eval_df["pred"] > 0.5).astype(int)

print(f"Accuracy  : {accuracy_score(eval_df['dropout'], eval_df['predicted_dropout']):.4f}")
print(f"Precision : {precision_score(eval_df['dropout'], eval_df['predicted_dropout']):.4f}")
print(f"Recall    : {recall_score(eval_df['dropout'], eval_df['predicted_dropout']):.4f}")
print(f"F1 Score  : {f1_score(eval_df['dropout'], eval_df['predicted_dropout']):.4f}")
print(f"AUC       : {roc_auc_score(eval_df['dropout'], eval_df['pred']):.4f}")

/opt/conda/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading processed data...
Training on Device: cuda...
Epoch 1 Loss: 0.4554
Epoch 2 Loss: 0.4408
Epoch 3 Loss: 0.4377

Evaluating Test Data...
Accuracy  : 0.8443
Precision : 0.8789
Recall    : 0.9317
F1 Score  : 0.9045
AUC       : 0.8333
